# Javis — Plano Bismarck
## Segundo cérebro operacional, local

Este caderno registra o contexto, as notas, as decisões e as ações do Plano Bismarck. O Javis analisa o estado registrado, aponta lacunas e riscos e sugere próximos passos para você revisar.

**Privacidade:** as análises são feitas por um modelo local no Ollama. O caderno conversa somente com `localhost` no próprio ThinkPad. Não há chave de API nem chamada à OpenAI ou a outro serviço de IA na nuvem neste caderno.


## Preparação única

1. Instale o Ollama para Windows pelo [site oficial](https://ollama.com/download/windows).
2. Abra o PowerShell e baixe o modelo local:

   `ollama pull qwen3:4b`

   O arquivo do modelo ocupa cerca de 2,5 GB. Esse download inicial exige internet; as notas do Plano Bismarck não são enviadas junto com ele.
3. Execute **Iniciar_Javis.ps1** na pasta deste caderno. Na primeira vez, ele instala o JupyterLab em um ambiente isolado.
4. No navegador, abra **Javis_Plano_Bismarck.ipynb** e escolha **Run → Run All Cells**.

Depois que Ollama, o modelo e JupyterLab estiverem instalados, o uso do Javis pode ser feito sem internet.


## Onde ficam os dados

O estado do plano é salvo em `%LOCALAPPDATA%\JavisBismarck\estado_plano_bismarck.json`, uma pasta local do Windows fora da pasta Documentos. O caderno e as instruções não armazenam suas notas.

Cada análise pede confirmação. O conteúdo é enviado pelo caderno somente ao serviço local do Ollama em `http://localhost:11434`, no mesmo ThinkPad. Não cole informações privadas em serviços externos se estiver usando outro programa.


## 1. Inicialização e armazenamento local

Execute esta célula primeiro. Ela cria a pasta local do Javis, se necessário.


In [4]:
from pathlib import Path
from datetime import datetime
import json
import os
import urllib.error
import urllib.request

PASTA_LOCAL = Path(os.environ.get("LOCALAPPDATA", Path.home() / "AppData" / "Local"))
PASTA_DADOS = PASTA_LOCAL / "JavisBismarck"
ARQUIVO_ESTADO = PASTA_DADOS / "estado_plano_bismarck.json"
OLLAMA_URL = "http://localhost:11434/api/chat"
MODELO_LOCAL = "qwen3:4b"

def _agora():
    return datetime.now().astimezone().isoformat(timespec="seconds")

def _estado_inicial():
    return {
        "projeto": "Plano Bismarck",
        "visao": "Javis como segundo cérebro operacional: registra contexto, analisa, sugere próximos passos e atualiza decisões confirmadas pela pessoa.",
        "objetivo": "",
        "notas": [],
        "decisoes": [],
        "acoes": [],
        "analises": [],
        "historico": []
    }

def _salvar_estado():
    PASTA_DADOS.mkdir(parents=True, exist_ok=True)
    temporario = ARQUIVO_ESTADO.with_suffix(".tmp")
    temporario.write_text(
        json.dumps(estado, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    temporario.replace(ARQUIVO_ESTADO)

def _carregar_estado():
    PASTA_DADOS.mkdir(parents=True, exist_ok=True)
    if not ARQUIVO_ESTADO.exists():
        return _estado_inicial()
    salvo = json.loads(ARQUIVO_ESTADO.read_text(encoding="utf-8"))
    inicial = _estado_inicial()
    inicial.update(salvo)
    return inicial

def _novo_id(prefixo, registros):
    numeros = []
    for item in registros:
        try:
            numeros.append(int(item["id"].split("-")[-1]))
        except (KeyError, ValueError, TypeError):
            pass
    return f"{prefixo}-{max(numeros, default=0) + 1:03d}"

def _registrar_historico(texto):
    estado["historico"].append({"data": _agora(), "evento": texto})

estado = _carregar_estado()
print("Javis local inicializado.")
print(f"Dados privados neste computador: {ARQUIVO_ESTADO}")


Javis local inicializado.
Dados privados neste computador: C:\Users\Lenovo\AppData\Local\JavisBismarck\estado_plano_bismarck.json


## 2. Capturar e atualizar o plano

Registre notas livremente. Decisões e ações são gravadas somente quando você chama explicitamente as funções de aprovação.


In [5]:
def definir_objetivo(texto):
    texto = texto.strip()
    if not texto:
        raise ValueError("Escreva um objetivo antes de salvar.")
    estado["objetivo"] = texto
    _registrar_historico("Objetivo do Plano Bismarck atualizado.")
    _salvar_estado()
    print("Objetivo atualizado.")

def registrar_nota(texto, origem="", tipo="ideia"):
    texto = texto.strip()
    if not texto:
        raise ValueError("A nota não pode ficar vazia.")
    item = {
        "id": _novo_id("N", estado["notas"]),
        "data": _agora(),
        "texto": texto,
        "origem": origem,
        "tipo": tipo,
        "analisada": False
    }
    estado["notas"].append(item)
    _registrar_historico(f"Nota {item['id']} registrada.")
    _salvar_estado()
    print(f"Nota registrada: {item['id']}")

def aprovar_decisao(texto, motivo="", substitui_id=None):
    texto = texto.strip()
    if not texto:
        raise ValueError("Descreva a decisão confirmada.")
    if substitui_id:
        anterior = next(
            (d for d in estado["decisoes"] if d["id"] == substitui_id),
            None
        )
        if anterior is None:
            raise ValueError(f"Não encontrei a decisão {substitui_id}.")
        anterior["status"] = "substituída"
    item = {
        "id": _novo_id("D", estado["decisoes"]),
        "data": _agora(),
        "texto": texto,
        "motivo": motivo,
        "status": "vigente",
        "substitui": substitui_id
    }
    estado["decisoes"].append(item)
    _registrar_historico(f"Decisão {item['id']} confirmada.")
    _salvar_estado()
    print(f"Decisão registrada: {item['id']}")

def registrar_acao(texto, prazo="", prioridade="média", origem=""):
    texto = texto.strip()
    if not texto:
        raise ValueError("Descreva a ação aprovada.")
    item = {
        "id": _novo_id("A", estado["acoes"]),
        "data": _agora(),
        "texto": texto,
        "prazo": prazo,
        "prioridade": prioridade,
        "status": "a fazer",
        "origem": origem
    }
    estado["acoes"].append(item)
    _registrar_historico(f"Ação {item['id']} registrada.")
    _salvar_estado()
    print(f"Ação registrada: {item['id']}")

def atualizar_acao(acao_id, novo_status):
    permitidos = {"a fazer", "em andamento", "concluída", "cancelada"}
    if novo_status not in permitidos:
        raise ValueError(f"Status permitido: {', '.join(sorted(permitidos))}")
    item = next((a for a in estado["acoes"] if a["id"] == acao_id), None)
    if item is None:
        raise ValueError(f"Não encontrei a ação {acao_id}.")
    item["status"] = novo_status
    item["atualizada_em"] = _agora()
    _registrar_historico(f"Ação {acao_id} atualizada para '{novo_status}'.")
    _salvar_estado()
    print(f"Ação {acao_id}: {novo_status}.")


## 3. Painel e análise local

O modelo local do Ollama é acessado apenas pelo endereço local do próprio computador. A análise fica no histórico local e não modifica decisões nem tarefas.


In [6]:
def ver_painel():
    abertas = [a for a in estado["acoes"] if a["status"] in {"a fazer", "em andamento"}]
    vigentes = [d for d in estado["decisoes"] if d["status"] == "vigente"]
    print(f"PLANO: {estado['projeto']}")
    print(f"Visão: {estado['visao']}")
    print(f"Objetivo: {estado['objetivo'] or 'Ainda não definido'}")
    print(f"Notas: {len(estado['notas'])} | Decisões vigentes: {len(vigentes)} | Ações abertas: {len(abertas)}")
    if abertas:
        print("\nAÇÕES ABERTAS")
        for acao in abertas:
            print(f"- {acao['id']} [{acao['prioridade']}] {acao['texto']} — {acao['status']}")

def analisar_plano(pergunta="Analise o estado atual e proponha os próximos passos."):
    print("A análise usará o contexto salvo e o enviará somente ao Ollama local neste computador.")
    print("Nenhuma decisão ou tarefa será alterada automaticamente.")
    confirmar = input("Confirmar análise local? Digite S para continuar: ").strip().lower()
    if confirmar not in {"s", "sim"}:
        print("Análise cancelada.")
        return None

    contexto = {k: v for k, v in estado.items() if k != "analises"}
    instrucoes = (
        "Você é Javis, o segundo cérebro operacional do Plano Bismarck. "
        "Responda em português. Separe fatos registrados, inferências e dúvidas. "
        "Analise contradições, lacunas, riscos e decisões possivelmente desatualizadas. "
        "Sugira próximos passos priorizados, explicando motivo e dependências. "
        "Diga o que precisa de confirmação humana. Não afirme que alterou registros. "
        "Sugestões nunca são decisões aprovadas."
    )
    entrada = (
        f"ESTADO REGISTRADO:\n{json.dumps(contexto, ensure_ascii=False, indent=2)}"
        f"\n\nPEDIDO:\n{pergunta}"
    )
    corpo = json.dumps({
        "model": MODELO_LOCAL,
        "stream": False,
        "messages": [
            {"role": "system", "content": instrucoes},
            {"role": "user", "content": entrada}
        ]
    }, ensure_ascii=False).encode("utf-8")
    pedido = urllib.request.Request(
        OLLAMA_URL,
        data=corpo,
        headers={"Content-Type": "application/json"},
        method="POST"
    )
    try:
        with urllib.request.urlopen(pedido, timeout=180) as resposta:
            dados = json.loads(resposta.read().decode("utf-8"))
    except urllib.error.HTTPError as erro:
        detalhe = erro.read().decode("utf-8", errors="replace")
        print(f"Ollama retornou erro ({erro.code}): {detalhe}")
        return None
    except urllib.error.URLError:
        print(
            "O Ollama local não respondeu. Abra o Ollama e confirme que "
            f"o modelo {MODELO_LOCAL} foi baixado."
        )
        return None

    texto = dados.get("message", {}).get("content", "").strip()
    if not texto:
        print("O modelo local não retornou texto.")
        return None
    estado["analises"].append({
        "id": _novo_id("AN", estado["analises"]),
        "data": _agora(),
        "pergunta": pergunta,
        "resposta": texto
    })
    _registrar_historico("Análise local do Javis recebida e salva.")
    _salvar_estado()
    print(texto)
    return texto

ver_painel()


PLANO: Plano Bismarck
Visão: Javis como segundo cérebro operacional: registra contexto, analisa, sugere próximos passos e atualiza decisões confirmadas pela pessoa.
Objetivo: Ainda não definido
Notas: 0 | Decisões vigentes: 0 | Ações abertas: 0


## 4. Uso rápido

Adicione uma célula de código pelo botão **+**, escreva uma linha e pressione **Shift+Enter**:

- `registrar_nota("Sua nota", origem="reunião", tipo="ideia")`
- `analisar_plano("Quais são os próximos passos e o que está em aberto?")`
- `definir_objetivo("Objetivo do Plano Bismarck")`
- `aprovar_decisao("Decisão confirmada", motivo="...", substitui_id="D-001")`
- `registrar_acao("Ação aprovada", prazo="AAAA-MM-DD", prioridade="alta", origem="D-001")`
- `atualizar_acao("A-001", "concluída")`

Ao corrigir uma decisão, registre a nova e informe o ID que ela substitui. A anterior continua no histórico.


In [ ]:
Set-ExecutionPolicy -Scope Process -ExecutionPolicy Bypass
& "C:\Users\Lenovo\Documents\Codex\2026-10-19\e\outputs\Iniciar_Javis.ps1"
